# core

> Fill in a module description here

In [1]:
# | default_exp search

In [2]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [3]:
# | export
from typing import Any, List, Sequence, Set, Optional, Dict, Coroutine, Tuple
from product_horse.db import SqlModelDatabase, Utterance, AbstractDatabase
from pprint import pprint
import uuid
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.schema import (
    TextNode,
    NodeRelationship,
    RelatedNodeInfo,
    BaseNode,
)
from product_horse.db import Transcription
from product_horse.extraction import AIModelClient, Questions, ModelType
from llama_index.core.vector_stores.types import VectorStoreQueryResult
import numpy as np
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.indices.base import BaseIndex
from llama_index.core.data_structs import IndexDict
from llama_index.core.schema import QueryBundle, NodeWithScore
from llama_index.core.storage.docstore.types import RefDocInfo
from llama_index.core.callbacks.base import CallbackManager
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.vector_stores.types import (
    VectorStore,
    VectorStoreQuery,
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
    FilterCondition
)
from llama_index.core.storage.storage_context import StorageContext

In [4]:
PG_URL = "postgresql://localhost:5432/product_horse"
db = SqlModelDatabase(database_url=PG_URL)
db.operations.run_migrations()

In [5]:
users = db.get_all_users()
transcripts = db.get_all_unique_transcriptions(mode="file_name")

In [6]:
pprint(transcripts[0].model_dump())

{'created_at': datetime.datetime(2024, 4, 23, 5, 28, 17, 503371),
 'embedded': True,
 'file_id': UUID('d91d3704-e8c7-4752-83a9-d671d24bd3bb'),
 'id': UUID('997c9bc8-f543-4a88-b85b-6ac4600549d4'),
 'text': 'Speaker A: Bit about your company and your role at it.\n'
         "Speaker B: Yeah, sure. So my dad, he's been painting forever, like "
         "maybe 40 years. I've been helping him since I was twelve, but I, "
         'when I graduated high school, I know just straight to working with '
         "him full time. I'm 45 now. Over the years, we've had as many as "
         'eight people working for us, or just the two of us. Different times, '
         'different things. We usually do just residential, but we have done '
         'commercial. Okay. And we try to keep it, you know, smaller sample, '
         'because the more people you hire sometimes, the more it is that you '
         'got to deal with, so.\n'
         'Speaker A: Absolutely. Yeah. My, my wife runs a small busines

In [7]:
pprint(transcripts[0].utterances[0].model_dump())

{'confidence': 0.8684474999999999,
 'end': 295634,
 'id': UUID('0cf8338b-6352-4232-afac-28fc76c65a31'),
 'speaker': 'A',
 'start': 290564,
 'text': 'Painter.com. Very imaginative, email.',
 'transcription_id': UUID('997c9bc8-f543-4a88-b85b-6ac4600549d4')}


In [8]:
# ai_client = AIModelClient()

# question = "For the transcript provided, identify which of the speakers is the researcher. Output must be either A or B, categorical."

# schema = await ai_client.create_schema(question)

In [9]:
# from pprint import pprint

# pprint(schema.model_dump())

In [10]:
# | export
async def get_nodes_from_transcripts(
    transcripts: Sequence[Transcription],
) -> List[TextNode]:
    import asyncio

    """Get nodes from transcripts.
    Each node represents a single utterance in the transcript.
    Uses AI to detect the speaker role.
    The node's metadata contains the speaker, start_seconds, end_seconds, and confidence."""
    parent_nodes: List[TextNode] = []
    child_nodes: List[TextNode] = []
    # embed_model = OpenAIEmbedding()
    ai_client = AIModelClient(model_type=ModelType.GPT_3_5_TURBO)
    role_extract_schema = Questions(
        questions=[
            {
                "categories": ["A", "B"],
                "id": "1",
                "output_type": "category",
                "text": "Identify which of the speakers is the researcher in this transcript, returning whichever speaker they are in a two-person conversation — A or B.",
            }
        ]
    )
    tasks = [
        ai_client.extract_information(transcript.text, role_extract_schema)
        for transcript in transcripts
    ]
    # Run all ai_client calls concurrently and wait for all to complete
    results = await asyncio.gather(*tasks)
    for transcript, result in zip(transcripts, results):
        next_transcript_id = None
        next_node_id = None
        interviewer = result.answers[0].categories[0]

        for utterance in reversed(transcript.utterances):
            role = "interviewee" if interviewer != utterance.speaker else "interviewer"
            child_node = TextNode(
                text=utterance.text,
                id_=str(utterance.id),
                # embedding=embed_model.get_query_embedding(utterance.text),
                # ref_doc_id=str(transcript.id),
                metadata={
                    "transcript_id": str(transcript.id),
                    "speaker": role,
                    "start_seconds": utterance.start,
                    "end_seconds": utterance.end,
                    "confidence": utterance.confidence,
                },
            )

            if next_node_id is not None and next_transcript_id == transcript.id:
                child_node.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
                    node_id=next_node_id
                )
            else:
                if next_node_id is not None:
                    child_node.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
                        node_id=next_node_id
                    )
            next_node_id = child_node.id_
            next_transcript_id = transcript.id
            child_nodes.append(child_node)

    reversed_child_nodes = list(reversed(child_nodes))
    for i, node in enumerate(reversed_child_nodes):
        if i > 0:
            if NodeRelationship.NEXT in reversed_child_nodes[i - 1].relationships:
                node.relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(
                    node_id=reversed_child_nodes[i - 1].id_
                )
    all_nodes = [*reversed_child_nodes]
    return all_nodes

In [11]:
nodes = await get_nodes_from_transcripts(transcripts)
relationships = nodes[0].relationships
pprint(relationships)
# pprint(len(nodes))
# for node in nodes:
#     if NodeRelationship.PREVIOUS not in node.relationships:
#         pprint(node.text)

pprint(relationships[NodeRelationship.NEXT])

{<NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='0a91c0cc-99ae-41f5-beb7-a5aef6184ad5', node_type=None, metadata={}, hash=None)}
RelatedNodeInfo(node_id='0a91c0cc-99ae-41f5-beb7-a5aef6184ad5', node_type=None, metadata={}, hash=None)


API for loading data from transcripts

In [12]:
# | export
def embed_and_augment_nodes(
    nodes: List[TextNode],
) -> Coroutine[Any, Any, Sequence[BaseNode]]:
    """Augments and embeds nodes. Returns a couroutine for async work in the main thread."""
    transformations = [
        # TitleExtractor(nodes=5, llm=llm),
        # QuestionsAnsweredExtractor(questions=3, llm=llm),
        OpenAIEmbedding()
    ]

    pipeline = IngestionPipeline(
        transformations=transformations,
    )
    return pipeline.arun(nodes=nodes, in_place=False)

How to use:

In [13]:
nodes_unstored = await get_nodes_from_transcripts(transcripts[0:5])
nodes = await embed_and_augment_nodes(nodes_unstored)

In [14]:
node = nodes[0]
node

TextNode(id_='fa45b187-56bf-4fdf-b97e-ef29f02ae1bd', embedding=[-0.013525602407753468, -0.020731978118419647, 0.023434368893504143, -0.028416048735380173, -0.012897774577140808, 0.02138710394501686, -0.012495146133005619, -0.00024140675668604672, -0.03524027019739151, -0.0021001535933464766, 0.02877090871334076, 0.030654393136501312, -0.011130302213132381, -0.025577174499630928, -0.0128909507766366, 0.01590043120086193, 0.02144169807434082, 0.00025804078904911876, 0.009260465390980244, -0.05049922317266464, -0.017115142196416855, -0.011874142102897167, -0.031473301351070404, 0.016364477574825287, -0.019135111942887306, 0.0052000549621880054, 0.019094165414571762, -0.01891673542559147, -0.002912235679104924, -0.011232664808630943, -0.010714024305343628, 0.0018442452419549227, -0.0396350659430027, 0.006523953750729561, -0.004927086643874645, -0.013238986022770405, -0.015136118978261948, -0.020254282280802727, 0.01891673542559147, -0.008557571098208427, 0.006278282031416893, -0.0041457135

In [15]:
# import logging
# import sys

# logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
# logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

Parts of llamaindex that don't work
- JSONReader - didn't take notes on why
- Treeindex 
- KeywordTableSimpleRetriever + KeywordTableSimpleRetriever - refuses to retrieve things
- Standard vectorindex - real shitty output

Implement my own Index and Retriever from scratch.

Will use LLamaindex's RetrieverQueryEngine and Vector stores, still.

In [16]:
# | export
class ExtendedQueryBundle(QueryBundle):
    query_str: str
    transcription_ids: List[str]
    seconds_buffer: int
    similarity_top_k: int

    def __init__(self, query_str: str, transcription_ids: List[str], seconds_buffer: int = 10, similarity_top_k: int = 25):
        self.query_str = query_str
        self.transcription_ids = transcription_ids
        self.seconds_buffer = seconds_buffer
        self.similarity_top_k = similarity_top_k

class TranscriptRetriever(BaseRetriever):
    """Returned by TranscriptIndex.as_retriever(), and pass into RetrieverQueryEngine.
    Then use RetrieverQueryEngine.retrieve
    Retrieval logic:
    1. Find the most relevant utterance in the transcript
    2. Expand to the adjacent utterances to get nodes that contain the most relevant information until they stop being similar to the original.
    3. Merge any overlapping nodes
    5. Rerank then return to user"""

    def __init__(self, vector_store: VectorStore, **kwargs: Any):
        self.vector_store = vector_store
        super().__init__(**kwargs)

    def _check_query_result(
        self,
        result: VectorStoreQueryResult,
        node_to_check_against: Optional[NodeWithScore] = None,
    ) -> List[NodeWithScore]:
        """really just sorts by seconds right now, doesn't actually check similarily as it originally did."""
        if result.nodes is None:
            return []
        if result.similarities is None:
            # populate with 0s
            result.similarities = [0 for _ in result.nodes]

        sorted_nodes_with_scores: List[NodeWithScore] = sorted(
            [
                NodeWithScore(node=node, score=similarity)
                for node, similarity in zip(result.nodes, result.similarities)
            ],
            key=lambda node: node.node.metadata["start_seconds"],
        )
        if node_to_check_against:
            raise NotImplementedError("Not implemented yet")
        else:
            nodes_list = sorted_nodes_with_scores
        return nodes_list
    
    def _get_nodes_based_on_relationship(self, node: BaseNode, relationship: NodeRelationship) -> Optional[TextNode]:
        related_node = node.relationships.get(relationship)
        if related_node:
            related_node_refreshed = self.vector_store.client.get(ids=[related_node.node_id])
            import json
            relationships_dict = json.loads(related_node_refreshed['metadatas'][0]['_node_content'])['relationships']
            new_node = TextNode(
                    text=related_node_refreshed['documents'][0],
                    id_=related_node_refreshed['ids'][0],
                    metadata={
                        "transcript_id": related_node_refreshed['metadatas'][0]["transcript_id"],
                        "speaker": related_node_refreshed['metadatas'][0]["speaker"],
                        "start_seconds": related_node_refreshed['metadatas'][0]["start_seconds"],
                        "end_seconds": related_node_refreshed['metadatas'][0]["end_seconds"],
                        "confidence": related_node_refreshed['metadatas'][0]["confidence"],
                    }
                )
            if '3' in relationships_dict:
                new_node.relationships[NodeRelationship.NEXT] = RelatedNodeInfo(
                    node_id=relationships_dict['3']['node_id']
                )
            if '2' in relationships_dict:
                new_node.relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(
                    node_id=relationships_dict['2']['node_id']
                )
            return new_node

    
    def _get_adjacent_nodes(self, node: BaseNode, seconds_buffer: int = 10) -> List[BaseNode]:
        """Get adjacent nodes in the transcript within a specified time window."""
        adjacent_nodes: List[BaseNode] = [node]

        # Get the start and end times of the original node
        original_start_ms = node.metadata["start_seconds"]
        original_end_ms = node.metadata["end_seconds"]

        # Calculate the time window boundaries
        start_window_ms = original_start_ms - seconds_buffer * 1000
        end_window_ms = original_end_ms + seconds_buffer * 1000
        # Retrieve previous nodes within the time window
        prev_node_id = node.relationships.get(NodeRelationship.PREVIOUS)
        if prev_node_id:
            prev_node_with_metadata = self.vector_store.client.get(ids=[prev_node_id.node_id])
        while prev_node_id:
            prev_node = self._get_nodes_based_on_relationship(node=node, relationship=NodeRelationship.PREVIOUS)
            if prev_node:
                prev_node_metadata = prev_node.metadata
                if prev_node_metadata['end_seconds'] < start_window_ms:
                    break
                adjacent_nodes.insert(0, prev_node)
                prev_node_id = prev_node.relationships.get(NodeRelationship.PREVIOUS)
                node = prev_node
            else:
                break

        # Retrieve next nodes within the time window
        next_node_id = node.relationships.get(NodeRelationship.NEXT)
        while next_node_id:
            next_node = self._get_nodes_based_on_relationship(node=node, relationship=NodeRelationship.NEXT)
            if next_node:
                if next_node.metadata['start_seconds'] > end_window_ms:
                    break
                adjacent_nodes.append(next_node)
                next_node_id = next_node.relationships.get(NodeRelationship.NEXT)
                node = next_node
            else:
                break
        return adjacent_nodes

    def _get_all_transcript_nodes(
        self, node_with_score: NodeWithScore, seconds_buffer: int = 10
    ) -> List[NodeWithScore]:
        """Retrieve all nodes in a transcript if they have the same transcript_id and the node has a relationship, up to a certain # of seconds"""
        adjacent_nodes: List[NodeWithScore] = []
        current_node = node_with_score.node
        if len(current_node.relationships) > 0:
            adjacent_node_list = self._get_adjacent_nodes(current_node, seconds_buffer=seconds_buffer)
            adjacent_node_list = self._check_query_result(result=VectorStoreQueryResult(nodes=adjacent_node_list))
            if not adjacent_node_list:
                return adjacent_nodes  # []
            adjacent_nodes.extend(adjacent_node_list)
            return adjacent_nodes  # happy path
        else:
            return adjacent_nodes  # []

    def _retrieve(self, query_bundle: ExtendedQueryBundle) -> List[NodeWithScore]:
        """Retrieve nodes given query."""
        self.embed_model = OpenAIEmbedding()
        metadata_filters = MetadataFilters(
            condition=FilterCondition.OR,
            filters=[
                MetadataFilter(key="transcript_id", value=str(transcription_id))
                for transcription_id in query_bundle.transcription_ids
            ]
        )
        query_embedding = self.embed_model.get_query_embedding(query_bundle.query_str)
        query_bundle.embedding = query_embedding
        vector_store_query = VectorStoreQuery(
            query_str=query_bundle.query_str,
            query_embedding=query_bundle.embedding,
            similarity_top_k=query_bundle.similarity_top_k,
            filters=metadata_filters

        )
        result = self.vector_store.query(vector_store_query)
        nodes_list = self._check_query_result(result)

        all_relevant_nodes: List[NodeWithScore] = []
        seen_node_ids: Set[str] = set()
        for node_with_score in nodes_list:
            all_nodes_in_transcript = self._get_all_transcript_nodes(node_with_score, seconds_buffer=query_bundle.seconds_buffer)
            for node in all_nodes_in_transcript:
                if node.node.id_ not in seen_node_ids:
                    seen_node_ids.add(node.node.id_)
                    all_relevant_nodes.append(node)


        all_relevant_nodes.sort(
            key=lambda x: (
                x.node.metadata["transcript_id"],
                x.node.metadata["start_seconds"],
            )
        )
        return all_relevant_nodes

    async def _aretrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Asynchronously retrieve nodes given query.

        Implemented by the user.

        """
        return self._retrieve(query_bundle)

    ####################################
    ### Image Retrieval: Not implementing
    ####################################

    async def _aimage_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Async retrieve image nodes or documents given image.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")

    async def _atext_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Async retrieve image nodes or documents given query text.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")

    def _image_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Retrieve image nodes or documents given image.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")

    def _text_to_image_retrieve(
        self,
        query_bundle: QueryBundle,
    ) -> List[NodeWithScore]:
        """Retrieve image nodes or documents given query text.

        Implemented by the user.

        """
        raise NotImplementedError("Images type not supported in transcripts")


class TranscriptIndex(BaseIndex[IndexDict]):
    """For now this class is mainly going to be used as a pass-through.
    The indexing is done in a helper function and passed into a vector store first.
    API should look like:
    index = TranscriptIndex.from_vector_store(vector_store)
    query_engine = index.as_query_engine()
    Which returns RetrieverQueryEngine
    """

    def __init__(self, nodes: Sequence[BaseNode], **kwargs: Any):
        super().__init__(nodes, **kwargs)
        self._callback_manager = CallbackManager()

    @classmethod
    def from_vector_store(
        cls,
        vector_store: VectorStore,
        **kwargs: Any,
    ) -> "TranscriptIndex":
        """
        Initialize a TranscriptIndex object from a VectorStore.

        Args:
            vector_store (VectorStore): A VectorStore object.
            **kwargs: Additional keyword arguments.

        Returns:
            TranscriptIndex: A TranscriptIndex object.
        """
        if not vector_store.stores_text:
            raise ValueError(
                "Cannot initialize from a vector store that does not store text."
            )

        kwargs.pop("storage_context", None)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        return cls(
            nodes=[],
            storage_context=storage_context,
            **kwargs,
        )

    def _build_index_from_nodes(self, nodes: Sequence[BaseNode]) -> IndexDict:
        """Build the index from nodes."""
        index_dict = IndexDict()
        for node in nodes:
            index_dict.add_node(node)
        return index_dict

    def _insert(self, nodes: Sequence[BaseNode], **insert_kwargs: Any) -> None:
        """Index-specific logic for inserting nodes to the index struct."""
        raise NotImplementedError("Not implemented")

    def _delete_node(self, node_id: str, **delete_kwargs: Any) -> None:
        """Delete a node."""
        raise NotImplementedError("Not implemented")

    @property
    def ref_doc_info(self) -> Dict[str, RefDocInfo]:
        """Retrieve a dict mapping of ingested documents and their nodes+metadata.
        Original implementation borrowed from VectorStoreIndex"""
        if not self._vector_store.stores_text:
            node_doc_ids = list(self.index_struct.nodes_dict.values())
            nodes = self.docstore.get_nodes(node_doc_ids)

            all_ref_doc_info = {}
            for node in nodes:
                ref_node = node.source_node
                if not ref_node:
                    continue

                ref_doc_info = self.docstore.get_ref_doc_info(ref_node.node_id)
                if not ref_doc_info:
                    continue

                all_ref_doc_info: Dict[str, RefDocInfo] = {}
                all_ref_doc_info[ref_node.node_id] = ref_doc_info
            return all_ref_doc_info
        else:
            raise NotImplementedError(
                "Vector store integrations that store text in the vector store are "
                "not supported by ref_doc_info yet."
            )

    def as_retriever(self, **kwargs: Any) -> BaseRetriever:
        return TranscriptRetriever(
            callback_manager=self._callback_manager,
            # node_ids=self.index_struct.nodes_dict.keys(),
            vector_store=self.storage_context.vector_store,
            object_map=self._object_map,
            **kwargs,
        )

In [17]:
# | export
class SearchEngine:
    """Search engine for transcripts.
    Usage:
    search_engine = SearchEngine()
    utterances = search_engine.get_utterances_from_query(query, transcripts)"""

    def __init__(
        self, db: AbstractDatabase, path: str = "../chroma_db", seconds_buffer: int = 10, similarity_top_k: int = 25
    ):
        self.db = db
        self.client_id = "hardcoded-string-right-now"
        self.seconds_buffer = seconds_buffer
        self.similarity_top_k = similarity_top_k
        self.path = path
        self.vector_store: Optional[VectorStore] = self._create_client()

    async def _embed_and_augment_nodes(
        self, nodes: List[TextNode]
    ) -> Sequence[BaseNode]:
        "Embeds and augments nodes"
        return await embed_and_augment_nodes(nodes)

    def _create_client(self) -> ChromaVectorStore:
        """returns vector store. Creates vector store if they haven't been already."""
        chroma_client = chromadb.PersistentClient(path=self.path)
        chroma_collection = chroma_client.get_or_create_collection(self.client_id)
        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        return vector_store

    async def _check_and_add_nodes_from_transcriptions(
        self, transcripts: List[Transcription]
    ) -> Tuple[bool, str]:
        from uuid import UUID
        "Adds nodes to the vector store if they haven't been already. Returns a boolean and a string indicating the status of the operation."
        vector_store = self.vector_store
        if vector_store is None:
            raise ValueError("Vector store is None")
        transcripts_to_add: List[Transcription] = []
        nodes: List[TextNode] = []
        augmented_nodes: List[BaseNode] = []
        for transcript in transcripts:
            if transcript.embedded is False:
                transcripts_to_add.append(transcript)
        if len(transcripts_to_add) == 0:
            return True, "No nodes were added"  # early return if no need to add nodes
        nodes = await get_nodes_from_transcripts(transcripts_to_add)
        unique_transcript_ids: Set[str] = set()
        for node in nodes:
            unique_transcript_ids.add(node.metadata["transcript_id"])
        for transcript_id in unique_transcript_ids:
            if not self.db:
                raise ValueError("Database is None")
            self.db.update_transcription(UUID(transcript_id), {"embedded": True})
        augmented_nodes = await embed_and_augment_nodes(nodes)  # type: ignore
        vector_store = self._create_client()
        try:
            vector_store.add(augmented_nodes)
            return True, "all nodes added"
        except Exception as e:
            return False, str(e)

    def _make_query(
        self, query: str, transcriptions: List[Transcription]
    ) -> QueryBundle:
        "Makes a query bundle from a query and a list of transcripts"
        transcript_ids = [str(transcript.id) for transcript in transcriptions]
        return ExtendedQueryBundle(query_str=query, transcription_ids=transcript_ids)

    async def get_utterances_from_query(
        self, query: str, transcripts: Sequence[Transcription],
    ) -> Sequence[Utterance]:
        "Returns time-sorted utterances from a query"
        vector_store = self.vector_store
        if vector_store is None:
            raise ValueError("Vector store is None")
        try:
            status, message = await self._check_and_add_nodes_from_transcriptions(
                transcripts
            )
            if status is False:
                raise ValueError(message)
        except Exception as e:
            pprint(e)
        index = TranscriptIndex.from_vector_store(vector_store)

        query_engine = index.as_retriever()
        query_bundle: ExtendedQueryBundle = self._make_query(query, transcripts)
        query_bundle.seconds_buffer = self.seconds_buffer
        query_bundle.similarity_top_k = self.similarity_top_k
        result = query_engine.retrieve(query_bundle)
        utterance_ids = [node.node.id_ for node in result]
        utterances = self.db.get_utterances(utterance_ids)
        return utterances



In [19]:
search_engine = SearchEngine(seconds_buffer=10, similarity_top_k=5, db=db)
query = "Pricing thoughts"
transcripts: Sequence[Transcription] = db.get_all_unique_transcriptions(
    mode="file_name"
)
utterances = await search_engine.get_utterances_from_query(query, transcripts)



In [20]:
import json
# json.loads(search_engine.vector_store.client.get(ids=[str('a71089b8-e0f9-4cc8-a703-360d5c456c66')])['metadatas'][0]['_node_content'])['relationships']
print(len(utterances))
assert utterances[0].words is not None

19


In [21]:
for ut in utterances:
    if len(ut.words) != 0:
        print(ut.text)

Exactly. Yeah. Or if I had something that I knew and it like I tried it out, it was pretty close spot on to what I would do when I was breaking it down by myself and I could do it that way, then it would be something that I would more use. I just haven't found anything like that at this point.
Yeah, it sounds like maybe we're on the right track, but we still got a little ways to go before we're quite to what you're describing, unfortunately, but appreciate all the input on that. I guess maybe in terms of pricing then. I know this isn't the kind of thing you would buy right now, of course, given the limitations, but. Or I'm guessing maybe I'm wrong about that, but I think that we've been thinking about this in two different ways. We've been thinking about kind of a price for the 3d model, measurements and materialist piece. And then maybe like maybe you can get just the visualization separately. So maybe you can just get like the 3d model and the visualizer for some different price. I d

In [ ]:
transcripts: Sequence[Transcription] = db.get_all_unique_transcriptions(mode="file_name")
for transcript in transcripts:
    print(transcript.embedded)
# for transcript in transcripts:
#     db.update_transcription(transcript.id, {"embedded": False})



True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore